In [ ]:
import os, json, math, warnings

os.environ["TORCH_HOME"] = "./torch_hub"
os.makedirs("./torch_hub", exist_ok=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
warnings.filterwarnings("ignore")

SAVE_DIR = "./outputs"
FIG_DIR  = os.path.join(SAVE_DIR, "report_figures")   # all figures are written here
os.makedirs(FIG_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("working dir:", os.getcwd())
print("loading from:", SAVE_DIR)
print("figures will be saved to:", FIG_DIR)

In [ ]:
# Load saved artifacts (checkpoint / history / predictions / sample images)
CKPT_PATH = os.path.join(SAVE_DIR, "best_age_dinov2.pt")
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
cfg      = ckpt["config"]
AGE_MEAN = ckpt["age_mean"]
AGE_STD  = ckpt["age_std"]

history = ckpt.get("history")
if history is None:
    with open(os.path.join(SAVE_DIR, "history.json")) as f:
        history = json.load(f)

val_pred = pd.read_csv(os.path.join(SAVE_DIR, "val_predictions.csv"))
manifest = pd.read_csv(os.path.join(SAVE_DIR, "samples_manifest.csv"))
print("config:", cfg)
print("val predictions:", len(val_pred), "| sample images:", len(manifest))

In [ ]:
# Training / validation loss & MAE curves 
# Filename: report_figures/loss_curves.png
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history["train_loss"], marker="o", label="train")
ax[0].plot(history["val_loss"],   marker="o", label="val")
ax[0].set_title("Loss (Huber)"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss")
ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(history["train_mae"], marker="o", label="train")
ax[1].plot(history["val_mae"],   marker="o", label="val")
ax[1].set_title("MAE (years)"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("MAE")
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "loss_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Regression metrics on the validation set
# Filename: metrics.json  and  metrics_by_age_group.csv
y = val_pred["true_age"].values.astype(float)
p = val_pred["pred_age"].values.astype(float)
err = p - y
abserr = np.abs(err)

metrics = {
    "MAE_years":       float(abserr.mean()),
    "RMSE_years":      float(np.sqrt((err ** 2).mean())),
    "R2":              float(r2_score(y, p)),
    "Pearson_r":       float(np.corrcoef(y, p)[0, 1]),
    "bias_years":      float(err.mean()),          # mean(pred - true); >0 means over-estimation
    "within_5yr_pct":  float((abserr <= 5).mean() * 100),
    "within_10yr_pct": float((abserr <= 10).mean() * 100),
    "n_val":           int(len(y)),
}
with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("SAVED:", os.path.join(SAVE_DIR, "metrics.json"))
for k, v in metrics.items():
    print("  %-18s %s" % (k, ("%.4f" % v) if isinstance(v, float) else v))

# MAE per age group
bins   = [0, 10, 18, 25, 35, 45, 55, 65, 200]
labels = ["1-10", "11-18", "19-25", "26-35", "36-45", "46-55", "56-65", "65+"]
gdf = val_pred.copy()
gdf["age_group"] = pd.cut(gdf["true_age"], bins=bins, labels=labels)
by_group = gdf.groupby("age_group", observed=True)[["true_age", "pred_age"]].apply(
    lambda d: pd.Series({"n": len(d), "MAE": float(np.abs(d.pred_age - d.true_age).mean())})
)
by_group.to_csv(os.path.join(SAVE_DIR, "metrics_by_age_group.csv"))
print("SAVED:", os.path.join(SAVE_DIR, "metrics_by_age_group.csv"))
print(by_group)

In [ ]:
# Actual vs predicted age
# Left: predicted-vs-true scatter        
# Middle: residual histogram (pred - true)
# Right: MAE by age group (where the model is strong/weak)
# Filename: report_figures/pred_vs_true.png
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

ax[0].scatter(y, p, s=6, alpha=0.25)
ax[0].plot([0, 100], [0, 100], "r--", label="ideal")
ax[0].set_xlabel("True age"); ax[0].set_ylabel("Predicted age")
ax[0].set_title("Predicted vs True  (MAE=%.2f, R2=%.3f)" % (metrics["MAE_years"], metrics["R2"]))
ax[0].set_xlim(0, 100); ax[0].set_ylim(0, 100); ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].hist(err, bins=50, color="indianred")
ax[1].axvline(0, color="k", ls="--")
ax[1].set_title("Residuals (pred - true), bias=%.2f" % metrics["bias_years"])
ax[1].set_xlabel("error (years)"); ax[1].set_ylabel("count")

ax[2].bar([str(x) for x in by_group.index], by_group["MAE"], color="steelblue")
ax[2].set_title("MAE by age group"); ax[2].set_xlabel("age group"); ax[2].set_ylabel("MAE (years)")
ax[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "pred_vs_true.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Dataset age distribution =====
# Filename: report_figures/age_distribution.png
all_ages = pd.read_csv(os.path.join(SAVE_DIR, "all_ages.csv"))["age"].values
plt.figure(figsize=(8, 4))
plt.hist(all_ages, bins=50, color="seagreen")
plt.title("Age distribution (full dataset, n=%d)" % len(all_ages))
plt.xlabel("age"); plt.ylabel("count")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "age_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Rebuild the train model (for the feature map)
class AttentionPooling(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.attn  = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm  = nn.LayerNorm(dim)
    def forward(self, tokens):
        B = tokens.size(0)
        q = self.query.expand(B, -1, -1)
        pooled, _ = self.attn(q, tokens, tokens)
        return self.norm(pooled.squeeze(1))

class MLPHead(nn.Module):
    def __init__(self, in_dim, hidden=512, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x):
        return self.net(x)

class AgeRegressor(nn.Module):
    def __init__(self, backbone, embed_dim, num_heads, hidden, dropout):
        super().__init__()
        self.backbone = backbone
        self.pool = AttentionPooling(embed_dim, num_heads)
        self.head = MLPHead(embed_dim, hidden, dropout)
    def forward(self, x):
        feats = self.backbone.forward_features(x)
        pooled = self.pool(feats["x_norm_patchtokens"])
        return self.head(pooled).squeeze(-1)

backbone = torch.hub.load("facebookresearch/dinov2", cfg["model"], trust_repo=True)
model = AgeRegressor(backbone, backbone.embed_dim,
                     cfg["num_heads"], cfg["mlp_hidden"], cfg["dropout"]).to(device)
model.load_state_dict(ckpt["model"]); model.eval()

IMG_SIZE = cfg["img_size"]; RESIZE = int(round(IMG_SIZE * 1.15))
IMAGENET_MEAN = (0.485, 0.456, 0.406); IMAGENET_STD = (0.229, 0.224, 0.225)
eval_tf = transforms.Compose([transforms.Resize(RESIZE), transforms.CenterCrop(IMG_SIZE),
                              transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
disp_tf = transforms.Compose([transforms.Resize(RESIZE), transforms.CenterCrop(IMG_SIZE)])
print("model rebuilt and weights loaded.")

In [ ]:
# Feature-map visualization on random images =====
# pick the image randomly
# col 3: the input face
# col 2: PCA of DINOv2 patch tokens -> RGB  
# col 1: attention-pooling weights -> heatmap 
# Filename: report_figures/feature_maps.png

@torch.no_grad()
def analyze(model, x):
    feats = model.backbone.forward_features(x)
    tokens = feats["x_norm_patchtokens"]                                  # [1, N, C]
    q = model.pool.query.expand(tokens.size(0), -1, -1)
    pooled, attn_w = model.pool.attn(q, tokens, tokens,
                                     need_weights=True, average_attn_weights=True)
    pooled = model.pool.norm(pooled.squeeze(1))
    pred = model.head(pooled).squeeze(-1)                                 # [1]
    return tokens, attn_w, pred

sample_dir = os.path.join(SAVE_DIR, "sample_images")
pick = manifest.sample(n=min(6, len(manifest)))     # randomly chosen each run
fig, axes = plt.subplots(len(pick), 3, figsize=(9, 3 * len(pick)))
if len(pick) == 1:
    axes = axes[None, :]

for r, (_, row) in enumerate(pick.iterrows()):
    img = Image.open(os.path.join(sample_dir, row["filename"])).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(device)
    tokens, attn_w, pred = analyze(model, x)

    N = tokens.shape[1]; g = int(round(N ** 0.5))
    tok = tokens[0].float().cpu().numpy()
    pca = PCA(n_components=3).fit_transform(tok)
    pca = (pca - pca.min(0)) / (np.ptp(pca, 0) + 1e-8)
    pca_img = pca.reshape(g, g, 3)
    attn = attn_w[0, 0].float().cpu().numpy().reshape(g, g)
    pred_age = pred.item() * AGE_STD + AGE_MEAN

    axes[r, 0].imshow(disp_tf(img)); axes[r, 0].axis("off")
    axes[r, 0].set_title("pred %.0f / true %d" % (pred_age, int(row["true_age"])))
    axes[r, 1].imshow(pca_img); axes[r, 1].axis("off"); axes[r, 1].set_title("patch tokens (PCA-RGB)")
    axes[r, 2].imshow(attn, cmap="jet"); axes[r, 2].axis("off"); axes[r, 2].set_title("attention pooling")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "feature_maps.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:

print("Figures (in %s):" % os.path.abspath(FIG_DIR))
for fn in ["loss_curves.png", "pred_vs_true.png", "age_distribution.png", "feature_maps.png"]:
    print("  -", fn)
print("Tables (in %s):" % os.path.abspath(SAVE_DIR))
for fn in ["metrics.json", "metrics_by_age_group.csv"]:
    print("  -", fn)